# Wearable Devices Project: White Noise & Concentration
이 노트북은 웨어러블 디바이스 데이터를 분석하여 백색소음이 집중력에 도움이 되는지 머신러닝을 통해 이진 분류하는 프로젝트입니다.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 및 파일 경로 세팅
* **DATA_DIR**: 데이터가 위치한 폴더의 경로입니다.
  * 로컬 환경: `HiCardiEducationSave_230329`
  * 구글 코랩(Colab): 드라이브 마운트 후 드라이브 경로로 지정하거나 파일을 업로드한 경로를 작성하세요. (예: `"/content/HiCardiEducationSave_230329"`)
* **Train scenarios**: 1, 2, 3, 4회차 데이터를 학습 데이터로 사용합니다. (5회차 완전 배제)
* **Test scenarios**: 6, 7회차 데이터를 테스트 데이터로 사용합니다.

In [ ]:
# 데이터 폴더 경로 설정 (로컬 환경 및 구글 코랩 경로 호환용)
DATA_DIR = "HiCardiEducationSave_230329"

# 1. 파일 세팅 (5회차 완전 배제, Train: 1, 2, 3, 4회차 / Test: 6, 7회차)
train_scenarios = [
    {"base": "260603_공연1-1.txt", "task": "260603_백색소음1.txt", "label": 0, "session": "S1_1"},
    {"base": "260604_공연1-2.txt", "task": "260604_빠른음악1.txt", "label": 1, "session": "S1_2"},
    {"base": "260607_공연2-1.txt", "task": "260607_백색소음2.txt", "label": 0, "session": "S2_1"},
    {"base": "260608_공연2-2.txt", "task": "260608_빠른음악2.txt", "label": 1, "session": "S2_2"},
    {"base": "260609_공연3-1.txt", "task": "260609_백색소음3.txt", "label": 0, "session": "S3_1"},
    {"base": "260610_공연3-2.txt", "task": "260610_빠른음악3.txt", "label": 1, "session": "S3_2"},
    {"base": "260611_공연4-1.txt", "task": "260611_백색소음4.txt", "label": 0, "session": "S4_1"},
    {"base": "260613_공연4-2.txt", "task": "260613_빠른음악4.txt", "label": 1, "session": "S4_2"},
]

test_scenarios = [
    {"base": "260616_공연6-1.txt", "task": "260616_백색소음6.txt", "label": 0, "session": "S6_1"},
    {"base": "260617_공연6-2.txt", "task": "260617_빠른음악6.txt", "label": 1, "session": "S6_2"},
    {"base": "260618_공연7-1.txt", "task": "260618_백색소음7.txt", "label": 0, "session": "S7_1"},
    {"base": "260619_공연7-2.txt", "task": "260619_빠른음악7.txt", "label": 1, "session": "S7_2"},
]

## 2. 호흡 및 HRV 중심 피처 추출 함수
피부 온도를 배제하고 호흡수와 HRV의 변이도를 강화하여 10가지 핵심 피처를 추출합니다:
* **HRV 관련**: `Mean_HR`, `Mean_RR`, `SDNN`, `RMSSD`, `pNN50` (NN50 비율), `Range_RR` (RR 간격 범위)
* **호흡 관련**: `Mean_Resp` (평균 호흡수), `Std_Resp` (호흡수 표준편차)
* **ECG 원시신호**: `Raw_Std`, `Raw_Energy`

In [ ]:
def extract_features(filepath):
    try: 
        df = pd.read_csv(filepath, sep='\t', header=None)
    except: 
        return None

    # 기초 노이즈 제거 (1초 단위 중앙값 필터)
    df[1] = df[1].rolling(window=5, center=True, min_periods=1).median()
    df[2] = df[2].rolling(window=5, center=True, min_periods=1).median()
    df[4] = df[4].rolling(window=5, center=True, min_periods=1).median()

    window_rows, step_rows = 150, 50
    n_windows = (len(df) - window_rows) // step_rows + 1
    if n_windows <= 0: 
        return None

    windows_data = []
    for i in range(n_windows):
        start = i * step_rows
        w = df.iloc[start:start + window_rows]
        hr, rr = w[1].values, w[2].values
        resp = w[4].values

        true_beats = [rr[0]]
        for j in range(1, len(rr)):
            if rr[j] != rr[j-1]: 
                true_beats.append(rr[j])
        true_beats = np.array(true_beats)

        if len(true_beats) < 5: 
            continue
        diff_rr = np.diff(true_beats)
        raw = w.iloc[:, 11:75].values.flatten()
        raw = raw[~np.isnan(raw)]

        # New HRV/Respiration features
        nn50 = np.sum(np.abs(diff_rr) > 50) if len(diff_rr)>0 else 0
        pNN50 = nn50 / len(diff_rr) if len(diff_rr)>0 else 0
        range_rr = np.ptp(true_beats) if len(true_beats)>0 else 0

        windows_data.append({
            'Mean_HR': np.mean(hr),
            'Mean_RR': np.mean(true_beats),
            'SDNN': np.std(true_beats, ddof=1) if len(true_beats)>1 else 0,
            'RMSSD': np.sqrt(np.mean(diff_rr**2)) if len(diff_rr)>0 else 0,
            'pNN50': pNN50,
            'Range_RR': range_rr,
            'Raw_Std': np.std(raw) if len(raw)>0 else 0,
            'Raw_Energy': np.sum(raw**2)/len(raw) if len(raw)>0 else 0,
            'Mean_Resp': np.mean(resp),
            'Std_Resp': np.std(resp) if len(resp)>0 else 0
        })
    return pd.DataFrame(windows_data)

## 3. 1:1 베이스라인 차감(Delta) 정규화
태스크 상태의 값에서 베이스라인 상태(공연 데이터)의 평균을 차감하여 개인차를 보정합니다.

In [ ]:
def build_delta_matrix(scenarios):
    feature_cols = ['Mean_HR', 'Mean_RR', 'SDNN', 'RMSSD', 'pNN50', 'Range_RR', 'Raw_Std', 'Raw_Energy', 'Mean_Resp', 'Std_Resp']
    matrix_list = []

    for sc in scenarios:
        base_path = os.path.join(DATA_DIR, sc["base"])
        task_path = os.path.join(DATA_DIR, sc["task"])

        if not os.path.exists(base_path) or not os.path.exists(task_path):
            print(f"[경고] 파일 누락: {base_path} 또는 {task_path}")
            continue

        df_base = extract_features(base_path)
        df_task = extract_features(task_path)

        if df_base is not None and df_task is not None and not df_base.empty and not df_task.empty:
            base_means = df_base[feature_cols].mean()
            df_delta = df_task[feature_cols].copy()

            for col in feature_cols:
                df_delta[col] = df_delta[col] - base_means[col]

            df_delta['Label'] = sc["label"]
            matrix_list.append(df_delta)

    return pd.concat(matrix_list, ignore_index=True) if matrix_list else pd.DataFrame()

## 4. 파이프라인 가동 및 모델링 (SVM, Random Forest, Logistic Regression)

In [ ]:
print("--- 데이터 로드 및 전처리 가동 ---")
train_df = build_delta_matrix(train_scenarios)
test_df  = build_delta_matrix(test_scenarios)

if not train_df.empty and not test_df.empty:
    X_train = train_df.drop(['Label'], axis=1)
    y_train = train_df['Label']
    X_test  = test_df.drop(['Label'], axis=1)
    y_test  = test_df['Label']

    print(f"\n[데이터 확인]")
    print(f"Train 샘플 수: {len(X_train)}")
    print(f"Test 샘플 수: {len(X_test)}")

    # 데이터 스케일링
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 모델 1: SVM
    svm = SVC(kernel='rbf', random_state=42)
    svm.fit(X_train_scaled, y_train)
    svm_preds = svm.predict(X_test_scaled)

    # 모델 2: Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)

    # 모델 3: Logistic Regression
    lr = LogisticRegression(random_state=42)
    lr.fit(X_train_scaled, y_train)
    lr_preds = lr.predict(X_test_scaled)

    print(f"\n======================================")
    print(f"[최종 성적표] 10 Features (HRV+호흡) 기반")
    print(f"======================================")
    print(f"★ SVM 정확도: {accuracy_score(y_test, svm_preds):.4f}")
    print(classification_report(y_test, svm_preds, target_names=['백색소음(0)', '빠른음악(1)']))

    print(f"--------------------------------------")
    print(f"☆ Random Forest 정확도: {accuracy_score(y_test, rf_preds):.4f}")
    print(classification_report(y_test, rf_preds, target_names=['백색소음(0)', '빠른음악(1)']))

    print(f"--------------------------------------")
    print(f"☆ Logistic Regression 정확도: {accuracy_score(y_test, lr_preds):.4f}")
    print(classification_report(y_test, lr_preds, target_names=['백색소음(0)', '빠른음악(1)']))
else:
    print("[오류] 파일 로드에 실패했습니다. 폴더 내 파일명과 코드를 다시 한번 확인해주세요.")